# Nepali Corpus Collection & Tokenizer Pipeline (Model L)

**End-to-End Pipeline on Google Colab**:
1. **Step 1: Automated Collection (Sangraha Verified)** — Direct 340MB parquet streaming with PyArrow micro-batches, auto-stopping at 400M tokens.
2. **Step 2a: Manual Collection (Wikipedia Dump)** — Streaming XML parser with zero-RAM memory clearing.
3. **Step 2b: Manual Collection (Live Multi-Threaded News Crawler)** — Crawls 50k+ live Ratopati and Setopati articles in parallel to achieve >= 20% manual tokens.
4. **Step 3: Document-level Split** — 90% train / 5% val / 5% test streaming split.
5. **Step 4: Custom SentencePiece BPE Training** — 32,000 vocab BPE from scratch.
6. **Step 5: Intrinsic Tokenizer Benchmark** — Empirical comparison based on *Dwivedi & Gopalan (ACL 2026)*.

## 0. Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas pyarrow sentencepiece regex trafilatura tabulate requests beautifulsoup4

import json, hashlib, re, random, bz2, os, gc, sys, shutil, time, math, collections, requests
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import pyarrow.parquet as pq
import trafilatura
import sentencepiece as spm
from tabulate import tabulate

LANGUAGE = 'nepali'
DRIVE_BASE = '/content/drive/MyDrive/24KaratAu/nepali'
MANUAL_DIR = f'{DRIVE_BASE}/cleaned/manual'
NONWEB_DIR = f'{DRIVE_BASE}/cleaned/automated_nonweb'
WEB_DIR    = f'{DRIVE_BASE}/cleaned/automated_web'
SPLITS_DIR = f'{DRIVE_BASE}/splits'
MANIFEST_DIR = f'{DRIVE_BASE}/manifests'

for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR, SPLITS_DIR, MANIFEST_DIR]:
    os.makedirs(d, exist_ok=True)

CHARS_PER_TOKEN = 6
AUTO_TOKEN_TARGET = 400_000_000
AUTO_CHAR_TARGET = AUTO_TOKEN_TARGET * CHARS_PER_TOKEN
MIN_CHARS = 100
MIN_DEV_RATIO = 0.40
SHARD_SIZE = 5000

def devanagari_ratio(text):
    if not text: return 0.0
    return sum(1 for c in text if '\u0900' <= c <= '\u097F') / len(text)

def clean_web_text(text):
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def text_hash_int(text):
    h = hashlib.sha256(text.encode('utf-8')).hexdigest()
    return int(h[:16], 16)

# Resume registry
all_hashes = set()
existing_manual_chars = 0
existing_auto_chars = 0
existing_docs = 0

for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR]:
    for sf in Path(d).glob('*.jsonl'):
        with open(sf, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                try:
                    doc = json.loads(line)
                    h = doc.get('text_sha256', '')
                    if h:
                        all_hashes.add(int(h[:16], 16) if len(h) > 16 else int(h, 16))
                    existing_docs += 1
                    c = doc.get('clean_text_characters', len(doc.get('text', '')))
                    if doc.get('collection_type') == 'manual':
                        existing_manual_chars += c
                    else:
                        existing_auto_chars += c
                except: pass

if existing_docs > 0:
    print(f'RESUMING: {existing_docs:,} docs | auto: ~{existing_auto_chars//5:,} tok | manual: ~{existing_manual_chars//5:,} tok')
else:
    print('Fresh start ready.')

--- 
## Step 1: Sangraha Verified Automated Collection (~400M Tokens)

In [ ]:
auto_chars_so_far = existing_auto_chars
nonweb_docs = []
nonweb_shard_idx = len(list(Path(NONWEB_DIR).glob('*.jsonl')))
nonweb_total_chars = 0
nonweb_total_docs = 0

def flush_nonweb():
    global nonweb_docs, nonweb_shard_idx
    if not nonweb_docs: return
    sf = f'{NONWEB_DIR}/nonweb_shard_{nonweb_shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as f:
        for d in nonweb_docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  Shard saved to Drive: {Path(sf).name} ({len(nonweb_docs):,} docs)')
    nonweb_docs = []
    nonweb_shard_idx += 1
    gc.collect()

if auto_chars_so_far >= AUTO_CHAR_TARGET:
    print(f'Already at auto target (~{auto_chars_so_far//5:,} tokens)! Skipping Step 1.')
else:
    print('Fetching Sangraha verified Nepali Parquet file listing...')
    api_url = 'https://huggingface.co/api/datasets/ai4bharat/sangraha'
    req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp:
        repo_data = json.loads(resp.read().decode())
        all_files = [f.get('rfilename') for f in repo_data.get('siblings', [])]

    nep_files = sorted([f for f in all_files if f.startswith('verified/nep/') and f.endswith('.parquet') and 'data-0.parquet' not in f])
    print(f'Found {len(nep_files)} 340MB Parquet files.')

    for idx, rel_path in enumerate(nep_files):
        if auto_chars_so_far >= AUTO_CHAR_TARGET: break
        file_name = Path(rel_path).name
        temp_local = f'/content/{file_name}'
        dl_url = f'https://huggingface.co/datasets/ai4bharat/sangraha/resolve/main/{rel_path}'
        
        print(f'[{idx+1}/{len(nep_files)}] Downloading {file_name} (~340MB)...')
        urllib.request.urlretrieve(dl_url, temp_local)
        
        pf = pq.ParquetFile(temp_local)
        for batch in pf.iter_batches(batch_size=5000, columns=['text', 'type']):
            batch_df = batch.to_pandas()
            for row in batch_df.itertuples():
                doc_type = str(getattr(row, 'type', '')).strip().lower()
                text = str(getattr(row, 'text', ''))
                if not text or text == 'nan': continue
                cleaned = clean_web_text(text)
                if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
                
                hi = text_hash_int(cleaned)
                if hi in all_hashes: continue
                all_hashes.add(hi)
                
                nonweb_docs.append({
                    'doc_id': f'{hi:x}',
                    'source': 'sangraha', 'source_type': f'hf_sangraha_{doc_type}',
                    'collection_type': 'automated', 'language': LANGUAGE,
                    'text': cleaned, 'clean_text_characters': len(cleaned),
                    'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
                    'text_sha256': f'{hi:x}', 'sangraha_type': doc_type,
                })
                nonweb_total_chars += len(cleaned)
                nonweb_total_docs += 1
                auto_chars_so_far += len(cleaned)
                
                if len(nonweb_docs) >= SHARD_SIZE: flush_nonweb()
                if auto_chars_so_far >= AUTO_CHAR_TARGET: break
            del batch_df
            gc.collect()
            if auto_chars_so_far >= AUTO_CHAR_TARGET: break
            
        if os.path.exists(temp_local): os.remove(temp_local)
        gc.collect()
        
    flush_nonweb()
    print(f'Step 1 Complete: {nonweb_total_docs:,} docs in Drive.')

--- 
## Step 2a: Manual Collection — Wikipedia XML Dump

In [ ]:
WIKI_DUMP = '/content/newiki-latest-pages-articles.xml.bz2'
man_shard = len(list(Path(MANUAL_DIR).glob('wiki_shard_*.jsonl')))

if man_shard > 0:
    print(f'Wikipedia shards already exist in Drive ({man_shard} shards). Skipping download/parse.')
else:
    print('Downloading Nepali Wikipedia dump...')
    !wget -q --show-progress -O {WIKI_DUMP} \
        'https://dumps.wikimedia.org/newiki/latest/newiki-latest-pages-articles.xml.bz2'

    WIKI_MARKUP = [
        (re.compile(r'\{\{[^}]*\}\}'), ''),
        (re.compile(r'\[\[Category:[^\]]*\]\]'), ''),
        (re.compile(r'\[\[श्रेणी:[^\]]*\]\]'), ''),
        (re.compile(r'\[\[[^|\]]*\|([^\]]*)\]\]'), r'\1'),
        (re.compile(r'\[\[([^\]]*)\]\]'), r'\1'),
        (re.compile(r'\[https?://[^\s\]]*\s*([^\]]*)\]'), r'\1'),
        (re.compile(r'\[https?://[^\]]*\]'), ''),
        (re.compile(r"'{2,5}"), ''),
        (re.compile(r'<ref[^>]*>.*?</ref>', re.DOTALL), ''),
        (re.compile(r'<ref[^/>]*/>'), ''),
        (re.compile(r'<[^>]+>'), ''),
        (re.compile(r'&[a-zA-Z]+;'), ' '),
        (re.compile(r'\[\d+\]'), ''),
    ]
    TABLE_PAT = [
        (re.compile(r'^\s*\{\|.*$', re.MULTILINE), ''),
        (re.compile(r'^\s*\|\}.*$', re.MULTILINE), ''),
        (re.compile(r'^\s*\|-.*$', re.MULTILINE), ''),
        (re.compile(r'^\s*!.*$', re.MULTILINE), ''),
        (re.compile(r'^\|.*$', re.MULTILINE), ''),
    ]
    SKIP_SEC = {'सन्दर्भ','सन्दर्भहरू','बाह्य कडीहरू','बाह्य लिङ्कहरू','श्रेणी',
                'यो पनि हेर्नुहोस्','स्रोत','टिप्पणी',
                'references','external links','see also','notes'}

    def clean_wiki(text):
        for p,r in TABLE_PAT: text = p.sub(r, text)
        for p,r in WIKI_MARKUP: text = p.sub(r, text)
        lines, result, skip = text.split('\n'), [], False
        for line in lines:
            s = line.strip().lower()
            if s in SKIP_SEC: skip = True; continue
            if re.match(r'^={2,}.*={2,}$', s): skip = False
            if not skip: result.append(line)
        text = '\n'.join(result)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()

    with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
        header = f.read(5000)
        ns_match = re.search(r'xmlns="(http://www\.mediawiki\.org/xml/export-[^"]+)"', header)
        NS = '{' + ns_match.group(1) + '}' if ns_match else ''

    man_docs, man_chars, man_count = [], 0, 0
    print('Processing Nepali Wikipedia dump...')
    with bz2.open(WIKI_DUMP, 'rt', encoding='utf-8') as f:
        context = iter(ET.iterparse(f, events=('start', 'end')))
        event, root = next(context)
        for event, elem in context:
            tag = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
            if event != 'end' or tag != 'page': continue
            ns_el = elem.find(f'{NS}ns') if NS else elem.find('ns')
            if ns_el is None or ns_el.text != '0':
                elem.clear(); root.clear(); continue
            redirect = elem.find(f'{NS}redirect') if NS else elem.find('redirect')
            if redirect is not None:
                elem.clear(); root.clear(); continue
                
            title_el = elem.find(f'{NS}title') if NS else elem.find('title')
            id_el = elem.find(f'{NS}id') if NS else elem.find('id')
            title = title_el.text if title_el is not None else ''
            pid = id_el.text if id_el is not None else ''
            rev = elem.find(f'{NS}revision') if NS else elem.find('revision')
            raw = ''
            if rev is not None:
                te = rev.find(f'{NS}text') if NS else rev.find('text')
                if te is not None and te.text: raw = te.text
                
            elem.clear(); root.clear()
            if not raw.strip(): continue
            cleaned = clean_wiki(raw)
            if len(cleaned) < MIN_CHARS or devanagari_ratio(cleaned) < MIN_DEV_RATIO: continue
            hi = text_hash_int(cleaned)
            if hi in all_hashes: continue
            all_hashes.add(hi)
            man_docs.append({
                'doc_id': f'{pid}_{title}',
                'title': title, 'wiki_page_id': pid,
                'source': 'Nepali Wikipedia Dump', 'source_type': 'wikipedia_dump',
                'collection_type': 'manual', 'language': LANGUAGE,
                'text': cleaned, 'clean_text_characters': len(cleaned),
                'devanagari_ratio': round(devanagari_ratio(cleaned), 4),
                'text_sha256': f'{hi:x}',
            })
            man_chars += len(cleaned); man_count += 1
            if len(man_docs) >= SHARD_SIZE:
                sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
                with open(sf,'w',encoding='utf-8') as out:
                    for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
                print(f'  Wrote {Path(sf).name}: {len(man_docs):,} docs')
                man_docs = []; man_shard += 1; gc.collect()

    if man_docs:
        sf = f'{MANUAL_DIR}/wiki_shard_{man_shard:04d}.jsonl'
        with open(sf,'w',encoding='utf-8') as out:
            for d in man_docs: out.write(json.dumps(d,ensure_ascii=False)+'\n')
        print(f'  Wrote {Path(sf).name}: {len(man_docs):,} docs')

    print('Wiki Dump Processing Complete.')

--- 
## Step 2b: Manual Collection — High-Yield Live Multi-Threaded News Crawler

In [ ]:
print('Generating verified live URL lists...')
target_urls = []

# Ratopati (IDs from 585,000 downwards)
target_urls.extend([f'https://www.ratopati.com/story/{i}' for i in range(585000, 535000, -1)])

# Setopati across core categories
for cat in ['politics', 'social', 'kinmel', 'opinion', 'sports', 'art']:
    target_urls.extend([f'https://www.setopati.com/{cat}/{i}' for i in range(396500, 385000, -1)])

print(f'Total Target Candidate URLs: {len(target_urls):,}')

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

buffer_docs = []
shard_idx = len(list(Path(MANUAL_DIR).glob('manual_news_shard_*.jsonl')))
total_news_chars = 0
total_news_docs = 0

def flush_shard():
    global buffer_docs, shard_idx
    if not buffer_docs: return
    sf = f'{MANUAL_DIR}/manual_news_shard_{shard_idx:04d}.jsonl'
    with open(sf, 'w', encoding='utf-8') as f:
        for d in buffer_docs:
            f.write(json.dumps(d, ensure_ascii=False) + '\n')
    print(f'  Saved Shard to Drive: {Path(sf).name} ({len(buffer_docs):,} docs)')
    buffer_docs = []
    shard_idx += 1
    gc.collect()

def fetch_and_clean_url(url):
    try:
        r = requests.get(url, headers=headers, timeout=4)
        if r.status_code == 200:
            text = trafilatura.extract(r.text, include_tables=False)
            if text:
                clean_p = clean_web_text(text)
                if len(clean_p) >= MIN_CHARS and devanagari_ratio(clean_p) >= MIN_DEV_RATIO:
                    h = hashlib.sha256(clean_p.encode('utf-8')).hexdigest()
                    source_name = 'Ratopati' if 'ratopati' in url else 'Setopati'
                    return {
                        'doc_id': h[:16],
                        'url': url,
                        'source': source_name,
                        'source_type': 'news_web',
                        'collection_type': 'manual',
                        'language': 'nepali',
                        'text': clean_p,
                        'clean_text_characters': len(clean_p),
                        'devanagari_ratio': round(devanagari_ratio(clean_p), 4),
                        'text_sha256': h
                    }
    except: pass
    return None

print('\nStarting Parallel Multi-Threaded Crawl (30 workers)...')
start_time = time.time()

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(fetch_and_clean_url, u): u for u in target_urls}
    processed = 0
    
    for future in as_completed(futures):
        processed += 1
        doc = future.result()
        if doc:
            buffer_docs.append(doc)
            total_news_chars += doc['clean_text_characters']
            total_news_docs += 1
            if len(buffer_docs) >= SHARD_SIZE:
                flush_shard()
                
        if processed % 2000 == 0:
            elapsed = time.time() - start_time
            rate = processed / max(1, elapsed)
            print(f'  Processed: {processed:,} / {len(target_urls):,} URLs ({rate:.1f} urls/s) | Accepted Docs: {total_news_docs:,} | Chars: {total_news_chars:,} (~{total_news_chars//5:,} tokens)')

flush_shard()
print(f'\nCRAWL COMPLETE: Collected {total_news_docs:,} new manual documents ({total_news_chars:,} chars, ~{total_news_chars//5:,} tokens).')

--- 
## Step 3: Train / Validation / Test Splits

In [ ]:
random.seed(42)
out = {s: open(f'{SPLITS_DIR}/{s}.txt', 'w', encoding='utf-8') for s in ['train', 'val', 'test']}
jout = {s: open(f'{SPLITS_DIR}/{s}.jsonl', 'w', encoding='utf-8') for s in ['train', 'val', 'test']}
counts = {'train': 0, 'val': 0, 'test': 0}
manual_char_total, auto_char_total = 0, 0

for d in [MANUAL_DIR, NONWEB_DIR, WEB_DIR]:
    for sf in sorted(Path(d).glob('*.jsonl')):
        with open(sf, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                r = random.random()
                s = 'train' if r < 0.90 else ('val' if r < 0.95 else 'test')
                jout[s].write(line)
                doc = json.loads(line)
                t = doc.get('text', '')
                c = doc.get('clean_text_characters', len(t))
                if doc.get('collection_type') == 'manual':
                    manual_char_total += c
                else:
                    auto_char_total += c
                if t.strip():
                    out[s].write(t.strip() + '\n\n')
                counts[s] += 1

for f in out.values(): f.close()
for f in jout.values(): f.close()

tot = manual_char_total + auto_char_total
print('=' * 60)
print(f'DATASET SPLIT & BALANCED ACCOUNTING COMPLETE')
print(f'  Manual:    ~{manual_char_total // 5:,} tokens ({manual_char_total / max(1, tot) * 100:.1f}%)')
print(f'  Automated: ~{auto_char_total // 5:,} tokens ({auto_char_total / max(1, tot) * 100:.1f}%)')
print(f'  Total:     ~{tot // 5:,} tokens')
for s in ['train', 'val', 'test']: print(f'  {s}: {counts[s]:,} docs')
print('=' * 60)

--- 
## Step 4: Train SentencePiece BPE Tokenizer (32k)

In [ ]:
import sentencepiece as spm
MPREFIX = f'{DRIVE_BASE}/tokenizer/nepali_bpe_32k'
TRAIN_TXT = f'{SPLITS_DIR}/train.txt'
mb = os.path.getsize(TRAIN_TXT) / (1024 * 1024)
print(f'Training BPE on {mb:.1f} MB...')

spm.SentencePieceTrainer.train(
    input=TRAIN_TXT, model_prefix=MPREFIX, vocab_size=32000,
    model_type='bpe', character_coverage=0.9995, num_threads=2,
    max_sentence_length=8192, input_sentence_size=500_000,
    shuffle_input_sentence=True,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='<pad>', unk_piece='<unk>', bos_piece='<bos>', eos_piece='<eos>',
    split_by_unicode_script=True, split_by_whitespace=True,
    split_digits=True, byte_fallback=True,
)

sp = spm.SentencePieceProcessor()
sp.load(f'{MPREFIX}.model')
print(f'Vocab size: {sp.get_piece_size():,}')
print('Sample encoding:')
print(sp.encode('नेपाल एक सुन्दर, सार्वभौमसत्ता सम्पन्न राष्ट्र हो।', out_type=str))

--- 
## Step 5: Intrinsic Tokenizer Benchmark (Dwivedi & Gopalan, ACL 2026)

In [ ]:
VAL_TXT = f'{SPLITS_DIR}/val.txt'
std_bpe_path = f'{MPREFIX}.model'

COMBINING_MARKS = set(chr(c) for c in range(0x0901, 0x0904)) | \
                  set(chr(c) for c in range(0x093E, 0x094E)) | {chr(0x0962), chr(0x0963)}

def count_broken_graphemes(tokens):
    broken = 0
    for tok in tokens:
        clean_tok = tok.lstrip(' ')
        if clean_tok and clean_tok[0] in COMBINING_MARKS:
            broken += 1
    return broken

val_sentences = []
with open(VAL_TXT, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line and len(line) > 20: val_sentences.append(line)
        if len(val_sentences) >= 5000: break

sp = spm.SentencePieceProcessor()
sp.load(std_bpe_path)

total_words, total_tokens, total_chars, total_broken = 0, 0, 0, 0
token_freqs = collections.Counter()

for text in val_sentences:
    words = text.split()
    tokens = sp.encode(text, out_type=str)
    token_ids = sp.encode(text, out_type=int)
    total_words += len(words)
    total_tokens += len(tokens)
    total_chars += len(text)
    total_broken += count_broken_graphemes(tokens)
    token_freqs.update(token_ids)

fertility = total_tokens / max(1, total_words)
chars_per_tok = total_chars / max(1, total_tokens)
broken_rate = (total_broken / max(1, total_tokens)) * 100
probs = [c / total_tokens for c in token_freqs.values()]
shannon_entropy = -sum(p * math.log2(p) for p in probs if p > 0)
renyi_entropy = -math.log2(sum(p**2 for p in probs))

print('\n' + '=' * 75)
print('FINAL INTRINSIC TOKENIZER METRICS (Standard BPE 32k)')
print('=' * 75)
print(f'  Fertility:          {fertility:.2f} tokens / word')
print(f'  Characters / Token: {chars_per_tok:.2f}')
print(f'  Broken Matra Rate:  {broken_rate:.3f}%')
print(f'  Shannon Entropy:    {shannon_entropy:.2f} bits')
print(f'  Rényi Entropy:      {renyi_entropy:.2f} bits')
print('=' * 75)